# Title: AI-Enhanced Customer Complaint Escalation Prediction

# Section 1: Dataset Simulation, Cleaning, EDA and Decision Tree Baseline

**Goal:** predict, at the time a complaint is submitted, whether it will later be
escalated to a supervisor or higher-level support team, using structured customer
and complaint data plus features extracted from the complaint text by an LLM.

**Notebook sections (Member 1):** Dataset simulation → Data cleaning → EDA → Decision Tree

**Target variable: `escalation_status`**
- **0 – Not escalated:** handled by the initial customer service representative
  without supervisor involvement.
- **1 – Escalated:** transferred or referred to a supervisor or higher-level support team.

| Column | Type | Description / values | Source |
|---|---|---|---|
| `complaint_id` | ID | Unique key (not used as a model feature) | Simulated |
| `customer_tenure_months` | Numeric | Months as a customer | Simulated |
| `customer_type` | Categorical | new, regular, premium | Simulated |
| `previous_complaints` | Numeric | Past complaints by the customer | Simulated |
| `previous_escalations` | Numeric | Past complaints that were escalated | Simulated |
| `purchase_value` | Numeric | Value of the purchase (RM) | Simulated |
| `communication_channel` | Categorical | email, chat, phone, social_media | Simulated |
| `complaint_category` | Categorical | Customer-selected: refund, delivery, payment, product_quality | Simulated |
| `days_since_purchase` | Numeric | Days between purchase and complaint | Simulated |
| `product_type` | Categorical | electronics, clothing, home, services | Simulated |
| `complaint_text` | Text | Complaint description | GenAI-generated |
| `sentiment` | Categorical | negative, neutral, positive | LLM-extracted |
| `llm_complaint_category` | Categorical | Category inferred from the text: refund, delivery, payment, product_quality | LLM-extracted |
| `urgency_level` | Ordinal | 1 (very low) to 5 (very high) | LLM-extracted |
| `severity_level` | Ordinal | 1 (very low) to 5 (very high) | LLM-extracted |
| `frustration_level` | Ordinal | 1 (very low) to 5 (very high) | LLM-extracted |
| `escalation_status` | Target | 0 = not escalated, 1 = escalated | Simulated |

## 1.1 Dataset Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42          # shared random seed for the whole team
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

from pathlib import Path

N_UNIQUE = 1440          # unique complaints
N_DUPLICATES = 60        # exact duplicate rows -> 1500 raw rows in total
TARGET_RATE = 0.20       # approximate share of escalated complaints
MISSING_RATE = 0.04      # per column, in the columns listed below
MISSING_COLS = ["customer_tenure_months", "purchase_value",
                "days_since_purchase", "communication_channel", "product_type"]
OUT_DIR = Path("data")

rng = np.random.default_rng(SEED)
n = N_UNIQUE


def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std()


def to_scale_1_5(latent):
    """Rank-based mapping of a latent score to 1-5 (about 15/20/30/20/15 %)."""
    r = pd.Series(latent).rank(pct=True).to_numpy()
    return np.digitize(r, [0.15, 0.35, 0.65, 0.85]) + 1


complaint_id = np.array([f"C{i:04d}" for i in range(1, n + 1)])

customer_type = rng.choice(["new", "regular", "premium"], n, p=[0.30, 0.50, 0.20])
tenure = np.where(customer_type == "new", rng.integers(1, 13, n),
         np.where(customer_type == "regular", rng.integers(6, 61, n),
                  rng.integers(24, 121, n)))

previous_complaints = np.clip(rng.poisson(0.4 + tenure / 60, n), 0, 10)
previous_escalations = rng.binomial(previous_complaints, 0.25)   # always <= previous_complaints

product_type = rng.choice(["electronics", "clothing", "home", "services"], n,
                          p=[0.30, 0.25, 0.25, 0.20])
median_value = pd.Series(product_type).map(
    {"electronics": 900, "clothing": 150, "home": 350, "services": 200}).to_numpy()
purchase_value = np.clip(np.round(median_value * np.exp(rng.normal(0, 0.6, n)), 2), 10, 8000)

category_probs = {   # refund, delivery, payment, product_quality
    "electronics": [0.20, 0.25, 0.10, 0.45],
    "clothing":    [0.30, 0.30, 0.10, 0.30],
    "home":        [0.15, 0.40, 0.10, 0.35],
    "services":    [0.25, 0.05, 0.50, 0.20],
}
cats = ["refund", "delivery", "payment", "product_quality"]
complaint_category = np.array([rng.choice(cats, p=category_probs[p]) for p in product_type])

communication_channel = rng.choice(["email", "chat", "phone", "social_media"], n,
                                   p=[0.30, 0.30, 0.25, 0.15])

scale_days = pd.Series(complaint_category).map(
    {"delivery": 7, "refund": 20, "payment": 12, "product_quality": 45}).to_numpy()
days_since_purchase = np.clip(np.round(rng.exponential(scale_days)), 0, 365).astype(int)

# quick preview of the structured fields
preview = pd.DataFrame({
    "complaint_id": complaint_id, "customer_tenure_months": tenure,
    "customer_type": customer_type, "previous_complaints": previous_complaints,
    "previous_escalations": previous_escalations, "purchase_value": purchase_value,
    "communication_channel": communication_channel,
    "complaint_category": complaint_category,
    "days_since_purchase": days_since_purchase, "product_type": product_type})
display(preview.head())

In [ ]:
log_value = np.log(purchase_value)
log_days = np.log1p(days_since_purchase)

sev_latent = (0.6 * zscore(log_value)
              + 0.4 * np.isin(complaint_category, ["product_quality", "payment"])
              + rng.normal(0, 0.8, n))
urg_latent = (0.5 * (complaint_category == "delivery")
              + 0.4 * (customer_type == "premium")
              - 0.3 * zscore(log_days)
              + 0.3 * np.isin(communication_channel, ["phone", "social_media"])
              + rng.normal(0, 0.9, n))
fru_latent = (0.5 * zscore(previous_complaints)
              + 0.5 * zscore(previous_escalations)
              + 0.4 * (communication_channel == "social_media")
              + 0.3 * zscore(log_days)
              + rng.normal(0, 0.8, n))

hidden_severity = to_scale_1_5(sev_latent)
hidden_urgency = to_scale_1_5(urg_latent)
hidden_frustration = to_scale_1_5(fru_latent)

# escalation depends on structured fields + hidden factors + random noise
core = (0.30 * previous_escalations
        + 0.10 * previous_complaints
        + 0.45 * (customer_type == "premium")
        + 0.35 * (communication_channel == "social_media")
        + 0.20 * zscore(log_value)
        + 0.45 * (hidden_severity - 3)
        + 0.40 * (hidden_urgency - 3)
        + 0.45 * (hidden_frustration - 3)
        + rng.normal(0, 0.7, n))


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


lo, hi = -15.0, 10.0          # bisection: find the intercept that gives TARGET_RATE
for _ in range(60):
    mid = (lo + hi) / 2
    if sigmoid(core + mid).mean() > TARGET_RATE:
        hi = mid
    else:
        lo = mid
prob = sigmoid(core + (lo + hi) / 2)
escalation_status = (rng.random(n) < prob).astype(int)

df = preview.copy()
df["escalation_status"] = escalation_status
hidden = pd.DataFrame({
    "complaint_id": complaint_id,
    "hidden_severity": hidden_severity,
    "hidden_urgency": hidden_urgency,
    "hidden_frustration": hidden_frustration,
})
print("Escalation rate: %.1f%%" % (100 * df["escalation_status"].mean()))
display(df.head())